<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>DIFFERENTIABLE AUTOENCODNG NEURAL OPERATOR -  STATIC MAPPING</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import random
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor, '\n')

In [ ]:
# Set random seed for reproducibility
seed = 15
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
input_var    = 3                                        # INPUT VARIABLES {a(x, y), x, y}
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
neurons      = 128
epochs       = 100
batchsize    = 30 
in_channels  = 3                       # NO. OF INPUT CHANNELS
mid_channels = 32                      # WIDTH OF THE CNN LAYERS IN THE SPECTRAL CONVOLUTION
out_channels = 1                       # NO. OF OUTPUT CHANNELS

learn_rate   = 1e-2
step_epoch   = 5
decay_rate   = 0.750

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_256_0.vtk"
data_file    = "Grid_256_"
Nfile        = 1000                                 # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../../..")
directory    = os.getcwd()
path         = directory + "/Flow_Data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale       = 10.0
yscale       = 5.0
vort_max     = 4.4

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))/yscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname  = 'vort_z'
vort       = np.zeros((Nfile, x_dim, y_dim))

for i in range(Nfile):
    file_name = data_file + str(i) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data              = reader.GetOutput()  
    pointData         = data.GetPointData().GetArray(fieldname)    
    vort    [i, :, :] = np.reshape(vtk_to_numpy(pointData), (x_dim, y_dim))
    
    print ("*"*85)
vort = vort / vort_max

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS - DIANO
</div>


In [ ]:
def COMPLEX_MULTIPLY(input_data, weights):                             # COMPLEX WEIGHTS X FFT MODES 

    return torch.einsum("bixy,ioxy->boxy", input_data, weights)

class CNN_LAYERS (nn.Module):                                           # LIFITNG/ PROJECTING THE INPUT LAYERS

    def __init__(self, in_channel, mid_channel, out_channel):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels = in_channel, out_channels= mid_channel, kernel_size=1),
            nn.GELU(),
            
            nn.Conv2d(in_channels = mid_channel, out_channels = out_channel, kernel_size=1)
            #nn.GELU(),

            #nn.Conv2d(in_channels = mid_channel, out_channels = out_channel, kernel_size=1)

            # MAY INCLUDE DROPOUT LAYERS nn.Dropout(p=0.5)
            )

    def forward (self, x):
        output = self.main(x)        
        return output

class SPECTRAL_BLOCK (nn.Module):
    def __init__(self, in_channels, out_channels, x_modes, y_modes):
        super().__init__()    

        # INITIALISATION OF COMPLEX WEIGHT FUNCTIONS 
        self.out_channels = out_channels 
        self.in_channels  = in_channels 
        self.scale        =  1 / (self.in_channels + self.out_channels)
        self.modes1       = x_modes      #Number of Fourier modes to multiply, at most floor(N/2) + 1
        self.modes2       = y_modes
        self.x_weights    = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))
        self.y_weights    = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def forward (self, x):

        batchsize = x.shape[0]

        # TRANSFORM THE GIVEN BATCH TO A FOURIER SPACE - 2D REAL FFT COMPUTATION
        x_fft = torch.fft.rfft2(x)

        # RETANING ONLY THE DESIRED FOURIER MODES
        out_fft = torch.zeros(batchsize, self.out_channels,  x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat).to(processor)

        out_fft[:, :, :self.modes1, :self.modes2]  = COMPLEX_MULTIPLY (x_fft[:, :, :self.modes1, :self.modes2], self.x_weights)

        out_fft[:, :, -self.modes1:, :self.modes2] = COMPLEX_MULTIPLY (x_fft[:, :, -self.modes1:, :self.modes2], self.y_weights)

        # TRANSFORM THE GIVEN BATCH BACK TO THE PHYSICAL SPACE - 2D REAL IFFT COMPUTATION
        x = torch.fft.irfft2(out_fft, s=(x.size(-2), x.size(-1)))

        return x
    

class ENCODER (nn.Module):

    def __init__(self, input_var, x_modes, y_modes, width, x_grid, y_grid):
        super().__init__()
        self.x_grid   = x_grid
        self.y_grid   = y_grid
        self.x_modes  = x_modes
        self.y_modes  = y_modes
        self.width    = width
        #self.padding  = 9                                          # PAD THE DOMAIN IF THE INPUT IS NON-PERIODIC

        self.p        = nn.Linear (3, self.width)                   # INPUT CHANNEL IS 3: (a(x, y), x, y)
        
        self.conv0    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv1    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv2    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv3    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)

        self.mlp0     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp1     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp2     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp3     = CNN_LAYERS (self.width, self.width, self.width)

        self.w0       = nn.Conv2d  (self.width, self.width, 1)
        self.w1       = nn.Conv2d  (self.width, self.width, 1)
        self.w2       = nn.Conv2d  (self.width, self.width, 1)
        self.w3       = nn.Conv2d  (self.width, self.width, 1)

        self.q        = CNN_LAYERS (self.width, self.width * 4, 1)  # OUTPUT CHANNEL IS 1: u(x, y)
        self.reduce   = nn.AvgPool2d(1, stride = 2)
        self.norm     = nn.Tanh()

    def forward (self, x):
        batch = x.shape[0]
        xgrid = torch.Tensor(self.x_grid).reshape(1, x_dim, y_dim, 1).repeat(x.shape[0], 1, 1, 1)
        ygrid = torch.Tensor(self.y_grid).reshape(1, x_dim, y_dim, 1).repeat(x.shape[0], 1, 1, 1)
        grid = torch.cat((xgrid, ygrid), dim=-1).to(processor)
        x = torch.cat((x, grid), dim=-1)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)

        #x = F.pad(x, [0,self.padding, 0,self.padding])
        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))
        
        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))
        
        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))

        #x1 = self.conv3(x)
        #x1 = self.mlp3(x1)
        #x2 = self.w3(x)
        #x = x1 + x2
        #x = self.reduce (torch.squeeze(x))

        #x = x[..., :-self.padding, :-self.padding]
        x = self.q(x)
        #x = x.permute(0, 2, 3, 1)
        return self.norm(x)

class DECODER (nn.Module):

    def __init__(self, x_modes, y_modes, width):
        super().__init__()

        self.x_modes  = x_modes
        self.y_modes  = y_modes
        self.width    = width
        self.padding  = 9                                           # PAD THE DOMAIN IF THE INPUT IS NON-PERIODIC

        self.p        = nn.Linear (1, self.width)                   # INPUT CHANNEL IS 3: (a(x, y), x, y)

        self.conv0    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv1    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv2    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv3    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)

        self.mlp0     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp1     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp2     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp3     = CNN_LAYERS (self.width, self.width, self.width)

        self.w0       = nn.Conv2d  (self.width, self.width, 1)
        self.w1       = nn.Conv2d  (self.width, self.width, 1)
        self.w2       = nn.Conv2d  (self.width, self.width, 1)
        self.w3       = nn.Conv2d  (self.width, self.width, 1)

        self.q        = CNN_LAYERS (self.width, self.width * 4, 1)  # OUTPUT CHANNEL IS 1: u(x, y)
        self.upsamp   = nn.ConvTranspose2d(self.width, self.width, 2, stride = 2)

    def forward (self, x):
        #x = F.pad(x, [0,self.padding, 0,self.padding])
        x  = x.permute(0, 2, 3, 1)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)
        x  = self.upsamp(x)
        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x  = x1 + x2
        x  = F.gelu(x)

        x  = self.upsamp (x)
        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x  = x1 + x2
        x  = F.gelu(x)

        x  = self.upsamp (x)
        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x  = x1 + x2
        x  = F.gelu(x)

        #x  = self.upsamp (x)
        #x1 = self.conv3(x)
        #x1 = self.mlp3(x1)
        #x2 = self.w3(x)
        #x  = x1 + x2
        
        #x  = self.upsamp (x)
        #x  = x[..., :-self.padding, :-self.padding]
        x  = self.q(x)
        x  = x.permute(0, 2, 3, 1)

        return x

def reproducible_split(dataset, train_size, test_size, seed=1):
    generator = torch.Generator().manual_seed(seed)
    return random_split(dataset, [train_size, test_size], generator=generator)

def count_parameters(model):
    total_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            total_params += param.numel()
    return total_params


In [ ]:
# MESH SPATIAL DATA
x_grid = torch.Tensor(x).reshape(1, x_dim, y_dim, 1)
y_grid = torch.Tensor(y).reshape(1, x_dim, y_dim, 1)

# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort).to(processor)
vort_output  = torch.Tensor(vort).to(processor)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

vort_input_reshaped  = vort_input.reshape  (total_samples, x_dim, y_dim, 1)
vort_output_reshaped = vort_output.reshape (total_samples, x_dim, y_dim, 1)

dataset               = TensorDataset(vort_input_reshaped, vort_output_reshaped)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10, shuffle=False)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DIANO - TRAINING DATA
</div>

In [ ]:
x_modes      = 8                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)
y_modes      = 8                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)

vort_encoder = ENCODER  (input_var, x_modes, y_modes, mid_channels, x_grid, y_grid).to(processor)
vort_decoder = DECODER  (x_modes, y_modes, mid_channels).to(processor)

encoder_optim  = optim.Adam(vort_encoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
decoder_optim  = optim.Adam(vort_decoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)

encoder_scheduler  = optim.lr_scheduler.StepLR(encoder_optim, step_size=step_epoch, gamma=decay_rate)
decoder_scheduler  = optim.lr_scheduler.StepLR(decoder_optim, step_size=step_epoch, gamma=decay_rate)

Loss_Data   = torch.empty(size=(epochs+1, 1))
loss_func   = nn.MSELoss()

for epoch in range (epochs+1):
    total_loss = 0.0
    for batch_idx, (w_in, w_out) in enumerate(train_loader):
        encoder_output  = vort_encoder (w_in)
        decoder_output  = vort_decoder (encoder_output)
        batch_loss      = loss_func    (decoder_output, w_out) 

        encoder_optim.zero_grad()
        decoder_optim.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            encoder_optim.step()
            decoder_optim.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', encoder_optim.param_groups[0]['lr'])
    print ("*"*85)
    
    encoder_scheduler.step()
    decoder_scheduler.step()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(vort_encoder.state_dict(),   "DIANO_ENCODER.pt" )
torch.save(vort_decoder.state_dict(),   "DIANO_DECODER.pt" )
torch.save(Loss_Data  [0:epoch], "Loss_plot.pt"  )

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        DIANO - TESTING DATA
</div>


In [ ]:
# TEST ERROR
vort_encoder.eval()
vort_decoder.eval()

total_loss = 0.0
for batch_idx, (w_in, w_out) in enumerate(test_loader):
    encoder_output  = vort_encoder (w_in)
    decoder_output  = vort_decoder (encoder_output)
    batch_loss      = loss_func    (decoder_output, w_out) 

    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())